# Complementos de Investigação Operacional — capítulo 3: Problemas sequenciais
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/applied-operations-research-course/blob/main/code/python/notebooks/03_scheduling.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 3 of the course "Complements of Operational Research" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/applied-operations-research-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("applied-operations-research-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/applied-operations-research-course
    RAIZ = os.path.abspath(os.path.join("applied-operations-research-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 3.1 Problemas sequenciais: regra de Johnson (2 máquinas)

Exemplo [`ex03_1_johnson.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_1_johnson/ex03_1_johnson.py) · funções da UC: [`sequenciamento.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_1_johnson/sequenciamento.py)

```text
Reproduz os exemplos do deck 3.1 (regra de Johnson, 2 máquinas).

Gráfica, seis trabalhos (impressão, acabamento): pela ordem de chegada T = 53 h (ociosos 12 e 11);
Johnson D, E, A, F, B, C com T = 45 h (ociosos 4 e 3).
Força bruta: o mínimo das 720 sequências é 45 h (só D E A F B C e D E F A B C), média 53 h, pior 61 h.
Para resolver na aula: Exemplo 1 (nove trabalhos) T = 61, ociosos 11 e 2;
Exemplo 2 (sete trabalhos) T = 67, ociosos 1 e 17, e os empates não mudam o tempo total.
```

In [2]:
from sequenciamento import tempos, cmax, ocios, tempo_total, johnson, forca_bruta


def mostra(nome, seq, P):
    """Sequência, tempo total, ociosos e a tabela dos instantes de conclusão."""
    C = tempos(seq, P)
    print("  %s: %s  T = %d  ociosos = %s" % (nome, " ".join(seq), cmax(seq, P),
                                             ", ".join("%d" % x for x in ocios(seq, P))))
    for i in range(len(C[0])):
        print("    M%d:%s" % (i + 1, "".join("%4d" % C[k][i] for k in range(len(seq)))))


ok = True
print("Complementos de IO — deck 3.1: regra de Johnson (2 máquinas)")

# ------------------------------------------------------------ a gráfica (slide «Em Python»)
a = dict(A=8, B=11, C=5, D=3, E=4, F=10)
b = dict(A=9, B=5, C=2, D=5, E=12, F=9)
P = {j: (a[j], b[j]) for j in a}
print("\nGráfica (impressão, acabamento): somas %d e %d h" % (sum(a.values()), sum(b.values())))
mostra("ordem de chegada", list(P), P)
s = johnson(a, b)
mostra("Johnson", s, P)
print("  tempo_total(Johnson) = %d" % tempo_total(s, a, b))
ok &= cmax(list(P), P) == 53 and ocios(list(P), P) == [12, 11]
ok &= s == list("DEAFBC") and tempo_total(s, a, b) == 45 and ocios(s, P) == [4, 3]
ok &= [r[1] for r in tempos(s, P)] == [8, 20, 29, 38, 43, 45]

# ------------------------------------------------------------ força bruta: as 720 sequências
bf = forca_bruta(P)
vals = [v for v, _ in bf]
otimas = [q for v, q in bf if v == vals[0]]
print("\nForça bruta: %d sequências; mínimo %d (%s); média %.2f; pior %d"
      % (len(bf), vals[0], ", ".join(otimas), sum(vals) / len(vals), max(vals)))
hist = [(T, vals.count(T)) for T in range(min(vals), max(vals) + 1) if T in vals]
print("  n.º de sequências por T: " + ", ".join("%d: %d" % h for h in hist))
ok &= vals[0] == 45 and otimas == ["DEAFBC", "DEFABC"] and max(vals) == 61
ok &= round(sum(vals) / len(vals)) == 53 and hist[1] == (46, 38) and hist[5] == (50, 123)

# ------------------------------------------------------------ Exemplo 1: nove trabalhos
M1 = [2, 5, 4, 9, 6, 8, 7, 5, 4]
M2 = [6, 8, 7, 4, 3, 9, 3, 8, 11]
nomes = ["J%d" % k for k in range(1, 10)]
a1 = dict(zip(nomes, M1)); b1 = dict(zip(nomes, M2))
P1 = {j: (a1[j], b1[j]) for j in nomes}
print("\nExemplo 1 (nove trabalhos): somas %d e %d" % (sum(M1), sum(M2)))
s1 = johnson(a1, b1)
mostra("Johnson", s1, P1)
alt = "J1 J9 J3 J8 J2 J6 J4 J7 J5".split()
print("  com os empates escolhidos ao contrário: %s  T = %d" % (" ".join(alt), cmax(alt, P1)))
ok &= s1 == "J1 J3 J9 J2 J8 J6 J4 J5 J7".split() and cmax(s1, P1) == 61 and ocios(s1, P1) == [11, 2]
ok &= cmax(alt, P1) == 61

# ------------------------------------------------------------ Exemplo 2: sete trabalhos
M1 = [3, 12, 15, 6, 10, 11, 9]
M2 = [8, 10, 10, 6, 12, 1, 3]
nomes = [str(k) for k in range(1, 8)]
a2 = dict(zip(nomes, M1)); b2 = dict(zip(nomes, M2))
P2 = {j: (a2[j], b2[j]) for j in nomes}
print("\nExemplo 2 (sete trabalhos): somas %d e %d" % (sum(M1), sum(M2)))
s2 = johnson(a2, b2)
mostra("Johnson", s2, P2)
print("  empates (o 4 tem a = b = 6; o 2 e o 3 têm b = 10): outras sequências de Johnson")
variantes = ["1 5 4 2 3 7 6", "1 5 3 2 4 7 6", "1 5 2 3 4 7 6", "1 4 5 3 2 7 6"]
for v in variantes:
    print("    %s  T = %d" % (v, cmax(v.split(), P2)))
ok &= s2 == "1 4 5 2 3 7 6".split() and cmax(s2, P2) == 67 and ocios(s2, P2) == [1, 17]
ok &= all(cmax(v.split(), P2) == 67 for v in variantes)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 3.1: regra de Johnson (2 máquinas)

Gráfica (impressão, acabamento): somas 41 e 42 h
  ordem de chegada: A B C D E F  T = 53  ociosos = 12, 11
    M1:   8  19  24  27  31  41
    M2:  17  24  26  32  44  53
  Johnson: D E A F B C  T = 45  ociosos = 4, 3
    M1:   3   7  15  25  36  41
    M2:   8  20  29  38  43  45
  tempo_total(Johnson) = 45

Força bruta: 720 sequências; mínimo 45 (DEAFBC, DEFABC); média 52.99; pior 61
  n.º de sequências por T: 45: 2, 46: 38, 47: 15, 48: 17, 49: 17, 50: 123, 51: 41, 52: 96, 53: 76, 54: 58, 55: 51, 56: 46, 57: 30, 58: 58, 59: 20, 60: 20, 61: 12

Exemplo 1 (nove trabalhos): somas 50 e 59
  Johnson: J1 J3 J9 J2 J8 J6 J4 J5 J7  T = 61  ociosos = 11, 2
    M1:   2   6  10  15  20  28  37  43  50
    M2:   8  15  26  34  42  51  55  58  61
  com os empates escolhidos ao contrário: J1 J9 J3 J8 J2 J6 J4 J7 J5  T = 61

Exemplo 2 (sete trabalhos): somas 66 e 50
  Johnson: 1 4 5 2 3 7 6  T = 67  ociosos = 1, 17
    M1:   3   9  19  31

## 3.2 Problemas sequenciais: três ou mais máquinas

Exemplo [`ex03_2_m_machines.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_2_m_machines/ex03_2_m_machines.py) · funções da UC: [`sequenciamento.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_1_johnson/sequenciamento.py), [`m_maquinas.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_2_m_machines/m_maquinas.py)

```text
Reproduz os exemplos do deck 3.2 (três ou mais máquinas).

Gráfica com corte (impressão, corte, acabamento): min M1 = 3 >= máx. M2 = 3, a condição verifica-se;
G = (10, 14, 6, 5, 7, 11), H = (11, 8, 3, 7, 15, 10); Johnson D, E, A, F, B, C com T = 47 h
(ociosos 6, 35, 5); pela ordem de chegada 55 h; força bruta: mínimo 47 h.
Semana da plastificação: a condição falha (máx. M2 = 9, min M1 = 2, min M3 = 3);
Johnson «à força» 1, 3, 4, 2 com T = 39; ótimo (24 sequências) 3, 2, 4, 1 com T = 35.
Para resolver na aula: Exemplos 3 (T = 51), 4 (T = 52) e 5 (cinco máquinas, T = 51).
(A experiência com 500 problemas aleatórios está em ../03_3_tdm/experiencia_aleatoria.py.)
```

In [3]:
import numpy as np

from sequenciamento import tempos, cmax, ocios, forca_bruta
from m_maquinas import reduz, condicao, johnson_m


def mat2str(A):
    """Matriz/vetor no formato do mat2str do MATLAB (para as saídas serem iguais)."""
    A = np.atleast_2d(np.asarray(A, float))
    return "[" + ";".join(" ".join("%.15g" % x for x in linha) for linha in A) + "]"


def mostra(nome, seq, P):
    """Sequência, tempo total, ociosos e a tabela dos instantes de conclusão."""
    C = tempos(seq, P)
    print("  %s: %s  T = %d  ociosos = %s" % (nome, " ".join(seq), cmax(seq, P),
                                             ", ".join("%d" % x for x in ocios(seq, P))))
    for i in range(len(C[0])):
        print("    M%d:%s" % (i + 1, "".join("%4d" % C[k][i] for k in range(len(seq)))))


def analisa(titulo, linhas, nomes=None):
    """Condição, (G, H) e Johnson para a tabela linhas (uma linha por máquina)."""
    n = len(linhas[0])
    nomes = nomes or [str(k + 1) for k in range(n)]
    P = {nomes[k]: tuple(l[k] for l in linhas) for k in range(n)}
    verif, (m1, mm, meio) = condicao(P)
    G, H = reduz(P)
    print("\n" + titulo)
    print("  condição: min M1 = %d, min M%d = %d, máx. das intermédias = %d -> %s"
          % (m1, len(linhas), mm, meio, "verifica-se" if verif else "falha"))
    print("  G = %s,  H = %s" % (mat2str(list(G.values())), mat2str(list(H.values()))))
    s = johnson_m(P)
    mostra("Johnson", s, P)
    return P, verif, G, H, s


ok = True
print("Complementos de IO — deck 3.2: três ou mais máquinas")

# ------------------------------------------------------------ gráfica com corte
P, verif, G, H, s = analisa("Gráfica com corte (impressão, corte, acabamento)",
                            [[8, 11, 5, 3, 4, 10], [2, 3, 1, 2, 3, 1], [9, 5, 2, 5, 12, 9]], list("ABCDEF"))
mostra("ordem de chegada", list(P), P)
bf = forca_bruta(P)
print("  força bruta: mínimo das %d sequências = %d" % (len(bf), bf[0][0]))
ok &= verif and condicao(P)[1] == (3, 2, 3)
ok &= list(G.values()) == [10, 14, 6, 5, 7, 11] and list(H.values()) == [11, 8, 3, 7, 15, 10]
ok &= s == list("DEAFBC") and cmax(s, P) == 47 and ocios(s, P) == [6, 35, 5]
ok &= cmax(list(P), P) == 55 and bf[0][0] == 47

# ------------------------------------------------------------ plastificação: a condição falha
Q, verif, G, H, s = analisa("Semana da plastificação (impressão, plastificação, acabamento)",
                            [[2, 7, 2, 4], [4, 8, 9, 8], [3, 8, 7, 5]])
bf = forca_bruta(Q)
otimas = [q for v, q in bf if v == bf[0][0]]
print("  força bruta (%d sequências): ótimo %s" % (len(bf), ", ".join(otimas)))
mostra("ótimo", list(otimas[0]), Q)
ok &= not verif and condicao(Q)[1] == (2, 3, 9)
ok &= list(G.values()) == [6, 15, 11, 12] and list(H.values()) == [7, 16, 16, 13]
ok &= s == list("1342") and cmax(s, Q) == 39 and otimas == ["3241"] and bf[0][0] == 35

# ------------------------------------------------------------ para resolver na aula
P3, v3, G3, H3, s3 = analisa("Exemplo 3", [[8, 10, 6, 7, 11], [5, 6, 2, 3, 4], [4, 9, 8, 6, 5]])
P4, v4, G4, H4, s4 = analisa("Exemplo 4", [[6, 7, 9, 10, 6], [5, 6, 4, 3, 4], [11, 10, 6, 7, 8]])
P5, v5, G5, H5, s5 = analisa("Exemplo 5 (cinco máquinas)",
                             [[7, 6, 5, 8], [5, 6, 4, 3], [2, 4, 5, 3], [3, 5, 6, 2], [9, 10, 8, 6]])
ok &= v3 and condicao(P3)[1] == (6, 4, 6) and s3 == list("32145") and cmax(s3, P3) == 51
ok &= ocios(s3, P3) == [9, 31, 19]
ok &= v4 and condicao(P4)[1] == (6, 6, 6) and s4 == list("51234") and cmax(s4, P4) == 52
ok &= ocios(s4, P4) == [14, 30, 10]
ok &= v5 and condicao(P5)[1] == (5, 6, 6) and s5 == list("1324") and cmax(s5, P5) == 51
ok &= list(G5.values()) == [17, 21, 20, 16] and list(H5.values()) == [19, 25, 23, 14]
ok &= ocios(s5, P5) == [25, 33, 37, 35, 18]

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 3.2: três ou mais máquinas

Gráfica com corte (impressão, corte, acabamento)
  condição: min M1 = 3, min M3 = 2, máx. das intermédias = 3 -> verifica-se
  G = [10 14 6 5 7 11],  H = [11 8 3 7 15 10]
  Johnson: D E A F B C  T = 47  ociosos = 6, 35, 5
    M1:   3   7  15  25  36  41
    M2:   5  10  17  26  39  42
    M3:  10  22  31  40  45  47
  ordem de chegada: A B C D E F  T = 55  ociosos = 14, 43, 13
    M1:   8  19  24  27  31  41
    M2:  10  22  25  29  34  42
    M3:  19  27  29  34  46  55
  força bruta: mínimo das 720 sequências = 47

Semana da plastificação (impressão, plastificação, acabamento)
  condição: min M1 = 2, min M3 = 3, máx. das intermédias = 9 -> falha
  G = [6 15 11 12],  H = [7 16 16 13]
  Johnson: 1 3 4 2  T = 39  ociosos = 24, 10, 16
    M1:   2   4   8  15
    M2:   6  15  23  31
    M3:   9  22  28  39
  força bruta (24 sequências): ótimo 3241
  ótimo: 3 2 4 1  T = 35  ociosos = 20, 6, 12
    M1:   2   9  13  15
    M2:  11  19  27

## 3.3 Problemas sequenciais: método do desvio de tempo (TDM)

Exemplo [`ex03_3_tdm.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_3_tdm/ex03_3_tdm.py) · funções da UC: [`sequenciamento.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_1_johnson/sequenciamento.py), [`m_maquinas.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_2_m_machines/m_maquinas.py), [`desvio_tempo.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_3_tdm/desvio_tempo.py)

```text
Reproduz os exemplos do deck 3.3 (método do desvio de tempo, TDM).

Gráfica (2 máquinas): iteração 1 com B pela frente e E por trás; construída B, F, C, D, A, E;
TDM E, A, D, C, F, B com T = 46 h (Johnson 45); das 720 sequências, 40 dão 46 h ou menos.
Os três exemplos do capítulo (chegada, TDM, Johnson, ótimo): 53, 46, 45, 45; 55, 49, 47, 47; 39, 40, 39, 35.
Para resolver na aula: Exemplo 6 (= Exemplo 1 de 3.1) T = 63 contra 61 de Johnson;
Exemplo 7 (= Exemplo 5 de 3.2) T = 54 contra 51; Exemplo 8 T = 62 contra 60 (o ótimo).
(A experiência com 500 problemas aleatórios está em experiencia_aleatoria.py, só em Python.)
```

In [4]:
import numpy as np

from sequenciamento import tempos, cmax, ocios, forca_bruta
from m_maquinas import reduz, condicao, johnson_m
from desvio_tempo import desvios, tdm


def mat2str(A):
    """Matriz/vetor no formato do mat2str do MATLAB (para as saídas serem iguais)."""
    A = np.atleast_2d(np.asarray(A, float))
    return "[" + ";".join(" ".join("%.15g" % x for x in linha) for linha in A) + "]"


def mostra(nome, seq, P):
    """Sequência, tempo total, ociosos e a tabela dos instantes de conclusão."""
    C = tempos(seq, P)
    print("  %s: %s  T = %d  ociosos = %s" % (nome, " ".join(seq), cmax(seq, P),
                                             ", ".join("%d" % x for x in ocios(seq, P))))
    for i in range(len(C[0])):
        print("    M%d:%s" % (i + 1, "".join("%4d" % C[k][i] for k in range(len(seq)))))


def tabela(linhas, nomes=None):
    """Dicionário trabalho -> tempos a partir de uma tabela com uma linha por máquina."""
    n = len(linhas[0])
    nomes = nomes or [str(k + 1) for k in range(n)]
    return {nomes[k]: tuple(l[k] for l in linhas) for k in range(n)}


ok = True
print("Complementos de IO — deck 3.3: método do desvio de tempo (TDM)")

# ------------------------------------------------------------ gráfica, passo a passo
P = tabela([[8, 11, 5, 3, 4, 10], [9, 5, 2, 5, 12, 9]], list("ABCDEF"))
a = {j: t[0] for j, t in P.items()}
b = {j: t[1] for j, t in P.items()}
print("\nGráfica (impressão, acabamento); desvios (máquina, trabalho) em cada máquina")
s, constr = tdm(P, log=True)
print("  construída: %s;  TDM (invertida): %s" % (" ".join(constr), " ".join(s)))
mostra("TDM", s, P)
vals = [v for v, _ in forca_bruta(P)]
print("  das %d sequências, %d dão T <= %d" % (len(vals), sum(1 for v in vals if v <= 46), 46))
D1 = desvios(a, b, list(P))
ok &= D1["A"] == ((3, 1), (3, 0)) and D1["B"] == ((0, 0), (7, 6)) and D1["C"] == ((6, 0), (10, 3))
ok &= D1["E"] == ((7, 8), (0, 0)) and desvios(a, b, list("ACDF"))["F"] == ((0, 0), (0, 1))
ok &= constr == list("BFCDAE") and s == list("EADCFB") and cmax(s, P) == 46
ok &= [r[1] for r in tempos(s, P)] == [16, 25, 30, 32, 41, 46] and sum(1 for v in vals if v <= 46) == 40

# ------------------------------------------------------------ os três exemplos do capítulo
print("\nOs três exemplos do capítulo (tempos totais em horas)")
print("  %-36s %8s %5s %8s %6s" % ("", "chegada", "TDM", "Johnson", "ótimo"))
casos = [("Gráfica, 2 máquinas (3.1)", P),
         ("Com corte, 3 máquinas (3.2)",
          tabela([[8, 11, 5, 3, 4, 10], [2, 3, 1, 2, 3, 1], [9, 5, 2, 5, 12, 9]], list("ABCDEF"))),
         ("Plastificação, condição falha (3.2)", tabela([[2, 7, 2, 4], [4, 8, 9, 8], [3, 8, 7, 5]]))]
quadro = []
for nome, Q in casos:
    linha = [cmax(list(Q), Q), cmax(tdm(Q)[0], Q), cmax(johnson_m(Q), Q), forca_bruta(Q)[0][0]]
    quadro.append(linha)
    print("  %-36s %8d %5d %8d %6d" % ((nome,) + tuple(linha)))
sq, cq = tdm(casos[2][1])
print("  plastificação: TDM %s (construída %s)" % (" ".join(sq), " ".join(cq)))
ok &= quadro == [[53, 46, 45, 45], [55, 49, 47, 47], [39, 40, 39, 35]] and sq == list("2341")

# ------------------------------------------------------------ Exemplo 6 (= Exemplo 1 de 3.1)
E1 = tabela([[2, 5, 4, 9, 6, 8, 7, 5, 4], [6, 8, 7, 4, 3, 9, 3, 8, 11]], ["J%d" % k for k in range(1, 10)])
print("\nExemplo 6 (= Exemplo 1 de 3.1)")
s6, c6 = tdm(E1, log=True)
print("  construída: %s" % " ".join(c6))
mostra("TDM", s6, E1)
print("  Johnson: T = %d" % cmax(johnson_m(E1), E1))
ok &= c6 == "J4 J7 J5 J1 J3 J2 J8 J6 J9".split() and cmax(s6, E1) == 63 and ocios(s6, E1) == [13, 4]
ok &= cmax(johnson_m(E1), E1) == 61

# ------------------------------------------------------------ Exemplo 7 (= Exemplo 5 de 3.2)
E5 = tabela([[7, 6, 5, 8], [5, 6, 4, 3], [2, 4, 5, 3], [3, 5, 6, 2], [9, 10, 8, 6]])
G, H = reduz(E5)
print("\nExemplo 7 (= Exemplo 5 de 3.2): G = %s, H = %s" % (mat2str(list(G.values())), mat2str(list(H.values()))))
s7, c7 = tdm(E5, log=True)
print("  construída: %s" % " ".join(c7))
mostra("TDM", s7, E5)
mostra("Johnson", johnson_m(E5), E5)
ok &= c7 == list("4132") and s7 == list("2314") and cmax(s7, E5) == 54 and cmax(johnson_m(E5), E5) == 51

# ------------------------------------------------------------ Exemplo 8
E8 = tabela([[7, 8, 10, 9, 7], [5, 6, 4, 5, 7], [12, 10, 7, 8, 11]])
verif, info = condicao(E8)
G, H = reduz(E8)
print("\nExemplo 8: condição %s (min M1 = %d, min M3 = %d, máx. M2 = %d); G = %s, H = %s"
      % ("verifica-se" if verif else "falha", info[0], info[1], info[2],
         mat2str(list(G.values())), mat2str(list(H.values()))))
s8, c8 = tdm(E8, log=True)
print("  construída: %s" % " ".join(c8))
mostra("TDM", s8, E8)
mostra("Johnson", johnson_m(E8), E8)
print("  força bruta: ótimo = %d" % forca_bruta(E8)[0][0])
ok &= verif and info == (7, 7, 7) and c8 == list("34215") and s8 == list("51243") and cmax(s8, E8) == 62
ok &= johnson_m(E8) == list("12543") and cmax(johnson_m(E8), E8) == 60 and forca_bruta(E8)[0][0] == 60

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 3.3: método do desvio de tempo (TDM)

Gráfica (impressão, acabamento); desvios (máquina, trabalho) em cada máquina
  iteração 1 (máx. 11 e 12)
    A    (3, 1)  (3, 0)
    B    (0, 0)  (7, 6)
    C    (6, 0)  (10, 3)
    D    (8, 2)  (7, 0)
    E    (7, 8)  (0, 0)
    F    (1, 0)  (3, 1)
    pela frente: B;  por trás: E;  em construção: B _ _ _ _ E
  iteração 2 (máx. 10 e 9)
    A    (2, 1)  (0, 0)
    C    (5, 0)  (7, 3)
    D    (7, 2)  (4, 0)
    F    (0, 0)  (0, 1)
    pela frente: F;  por trás: A;  em construção: B F _ _ A E
  iteração 3 (máx. 5 e 5)
    C    (0, 0)  (3, 3)
    D    (2, 2)  (0, 0)
    pela frente: C;  por trás: D;  em construção: B F C D A E
  construída: B F C D A E;  TDM (invertida): E A D C F B
  TDM: E A D C F B  T = 46  ociosos = 5, 4
    M1:   4  12  15  20  30  41
    M2:  16  25  30  32  41  46
  das 720 sequências, 40 dão T <= 46

Os três exemplos do capítulo (tempos totais em horas)
                                        chegada

## 3.4 Problemas sequenciais: branch and bound

Exemplo [`ex03_4_branch_and_bound.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_4_branch_and_bound/ex03_4_branch_and_bound.py) · funções da UC: [`sequenciamento.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_1_johnson/sequenciamento.py), [`m_maquinas.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_2_m_machines/m_maquinas.py), [`desvio_tempo.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_3_tdm/desvio_tempo.py), [`bb_sequenciamento.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/03_scheduling/03_4_branch_and_bound/bb_sequenciamento.py)

```text
Reproduz os exemplos do deck 3.4 (branch and bound).

Plastificação (3 máquinas, tempo total): nó «3 primeiro» com C = (2, 11, 18) e LB = max(22, 34, 34) = 34;
raiz 32; árvore com 10 nós e ótimo 3, 2, 4, 1 com T = 35. Resumo: chegada 39, Johnson 39, TDM 40, B&B 35.
Impressora digital (1 máquina, atraso total): EDD N, M, L, K com atraso 9; limites ...K 4, ...L 7,
...LK 9, ...KL 7; árvore com 13 nós e ótimo N, M, K, L com atraso 7 (C = 3, 6, 8, 15; atrasos 0, 0, 0, 7).
Com 8 trabalhos: 8! = 40320 sequências e 109600 nós na árvore completa.
(A experiência com 100 problemas aleatórios está em experiencia_bb.py, só em Python.)
O TPC da Lista 3 não é resolvido aqui.
```

In [5]:
import math

from sequenciamento import tempos, cmax, forca_bruta
from m_maquinas import johnson_m
from desvio_tempo import tdm
from bb_sequenciamento import lb_fs, bb_fs, atraso, bb_atraso

ok = True
print("Complementos de IO — deck 3.4: branch and bound")

# ------------------------------------------------------------ parte 1: plastificação
Q = {"1": (2, 4, 3), "2": (7, 8, 8), "3": (2, 9, 7), "4": (4, 8, 5)}
print("\nParte 1 — plastificação (3 máquinas, tempo total)")
c3 = tempos(["3"], Q)[-1]
U = ["1", "2", "4"]
lbs = [c3[i] + sum(Q[j][i] for j in U) + min(sum(Q[j][i + 1:]) for j in U) for i in range(3)]
print("  nó «3 primeiro»: C = (%s), LB1, LB2, LB3 = %s -> LB = %d"
      % (", ".join("%d" % x for x in c3), ", ".join("%d" % x for x in lbs), lb_fs(["3"], Q)))
print("  raiz: LB = %d" % lb_fs([], Q))
(v, s), nos = bb_fs(Q)
print("  nós gerados (%d): %s" % (len(nos), ", ".join("%s: %d" % n for n in nos)))
print("  ótimo: %s com T = %d (força bruta: %d sequências, mínimo %d)"
      % (", ".join(s), v, len(forca_bruta(Q)), forca_bruta(Q)[0][0]))
ok &= c3 == [2, 11, 18] and lbs == [22, 34, 34] and lb_fs(["3"], Q) == 34 and lb_fs([], Q) == 32
ok &= (v, s) == (35, "3241") and len(nos) == 10
ok &= nos == [("1", 36), ("2", 39), ("3", 34), ("4", 36), ("31", 36), ("32", 35), ("34", 35),
              ("321", 36), ("324", 35), ("3241", 35)]

print("  resumo da semana da plastificação:")
resumo = [("ordem de chegada", list(Q)), ("Johnson sobre (G, H)", johnson_m(Q)),
          ("TDM sobre (G, H)", tdm(Q)[0]), ("branch and bound", list(s))]
for nome, seq in resumo:
    print("    %-22s %s  T = %d" % (nome, ", ".join(seq), cmax(seq, Q)))
ok &= [cmax(q, Q) for _, q in resumo] == [39, 39, 40, 35]
ok &= johnson_m(Q) == list("1342") and tdm(Q)[0] == list("2341")

# ------------------------------------------------------------ parte 2: impressora digital
p = {"K": 2, "L": 7, "M": 3, "N": 3}
d = {"K": 11, "L": 8, "M": 6, "N": 4}
print("\nParte 2 — impressora digital (1 máquina, atraso total); soma dos tempos = %d" % sum(p.values()))
edd = sorted(p, key=lambda j: d[j])
print("  EDD (prazo mais cedo primeiro): %s, atraso total = %d" % (", ".join(edd), atraso(edd, p, d)))
(v, s), nos = bb_atraso(p, d)
print("  nós gerados (%d): %s" % (len(nos), ", ".join("...%s: %d" % n for n in nos)))
Cj = [r[0] for r in tempos(list(s), {j: (p[j],) for j in p})]
print("  ótimo: %s com atraso %d;  C = (%s), atrasos %s"
      % (", ".join(s), v, ", ".join("%d" % x for x in Cj),
         ", ".join("%d" % max(0, c - d[j]) for c, j in zip(Cj, s))))
bfa = forca_bruta(p, lambda q: atraso(q, p, d))
print("  força bruta: %d sequências, mínimo %d" % (len(bfa), bfa[0][0]))
ok &= edd == list("NMLK") and atraso(edd, p, d) == 9 and (v, s) == (7, "NMKL") and len(nos) == 13
ok &= nos[:4] == [("K", 4), ("L", 7), ("M", 9), ("N", 11)]
ok &= dict(nos)["LK"] == 9 and dict(nos)["KL"] == 7 and dict(nos)["MKL"] == 7 and dict(nos)["NKL"] == 9
ok &= Cj == [3, 6, 8, 15] and bfa[0] == (7, "NMKL")

# ------------------------------------------------------------ tamanho da árvore completa
n = 8
print("\nCom %d trabalhos: %d sequências; a árvore completa tem %d nós"
      % (n, math.factorial(n), sum(math.perm(n, k) for k in range(1, n + 1))))
ok &= math.factorial(8) == 40320 and sum(math.perm(8, k) for k in range(1, 9)) == 109600

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 3.4: branch and bound

Parte 1 — plastificação (3 máquinas, tempo total)
  nó «3 primeiro»: C = (2, 11, 18), LB1, LB2, LB3 = 22, 34, 34 -> LB = 34
  raiz: LB = 32
  nós gerados (10): 1: 36, 2: 39, 3: 34, 4: 36, 31: 36, 32: 35, 34: 35, 321: 36, 324: 35, 3241: 35
  ótimo: 3, 2, 4, 1 com T = 35 (força bruta: 24 sequências, mínimo 35)
  resumo da semana da plastificação:
    ordem de chegada       1, 2, 3, 4  T = 39
    Johnson sobre (G, H)   1, 3, 4, 2  T = 39
    TDM sobre (G, H)       2, 3, 4, 1  T = 40
    branch and bound       3, 2, 4, 1  T = 35

Parte 2 — impressora digital (1 máquina, atraso total); soma dos tempos = 15
  EDD (prazo mais cedo primeiro): N, M, L, K, atraso total = 9
  nós gerados (13): ...K: 4, ...L: 7, ...M: 9, ...N: 11, ...LK: 9, ...MK: 11, ...NK: 13, ...KL: 7, ...ML: 9, ...NL: 11, ...MKL: 7, ...NKL: 9, ...NMKL: 7
  ótimo: N, M, K, L com atraso 7;  C = (3, 6, 8, 15), atrasos 0, 0, 0, 7
  força bruta: 24 sequências, mínimo 7

Com 8 trabalh